In [ ]:
import re
import sys
import numpy as np
import scipy.linalg as LA
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm

if '..' not in sys.path:
    sys.path = ['..'] + sys.path
from pfcommon import parse_sparse_matrix_file, parse_Amat_vars_file

In [ ]:
def compute_A(J, N_state_vars):
    Jfx = J[:N_state_vars, :N_state_vars]
    Jfy = J[:N_state_vars, N_state_vars:]
    Jgx = J[N_state_vars:, :N_state_vars]
    Jgy = J[N_state_vars:, N_state_vars:]
    Jgy_inv = LA.inv(Jgy)
    return Jfx - Jfy @ Jgy_inv @ Jgx

In [ ]:
base_dir = Path('../data/Sardinia/SM_configs_from_data/multiple_GFM/Genstat_01')
dirs = sorted([d for p in base_dir.glob('*') for d in p.glob('Ta_*') if p.is_dir()])
N_dirs = len(dirs)
print(f'Found {N_dirs} directories.')

In [ ]:
J, var_names, N_state_vars = [], [], []
for i in tqdm(range(N_dirs)):
    J.append(parse_sparse_matrix_file(dirs[i] / 'Jacobian.mtl'))
    var_names.append(parse_Amat_vars_file(dirs[i] / 'VariableToIdx_Amat.txt')[1])
    N_state_vars.append(len(var_names[-1]))

In [ ]:
A = []
for i in tqdm(range(N_dirs)):
    A.append(compute_A(J[i], N_state_vars[i]))

In [ ]:
eigvals = []
for i in tqdm(range(N_dirs)):
    eigvals.append(LA.eigvals(A[i]))

In [ ]:
max_eigvals = np.array([np.max(eig.real) for eig in eigvals])
thresh = 1e-6
unstable, = np.where(max_eigvals > thresh)
stable, = np.where(max_eigvals <= thresh)
print(f'{unstable.size} configurations have max eigenvalue > {thresh}.')

In [ ]:
fig,ax = plt.subplots(2, 1, figsize=(5,4), sharex=True, sharey=True)
x = 1 + np.arange(N_dirs // 2)
Ta = 4, 40
y0 = np.min(max_eigvals) / 10
for i, a in enumerate(ax):
    a.hlines(thresh, x[0], x[-1], color='tab:red', ls='--', lw=1)
    a.vlines(x,  y0, max_eigvals[i::2], ls=':', lw=0.5)
    a.plot(x, max_eigvals[i::2], 'ko', markerfacecolor='w', markersize=3, markeredgewidth=1)
    a.set_ylabel('max( Re{eig} )')
    a.set_title(r'$T_a = {}\,$s'.format(Ta[i]), fontsize=9)
    a.set_yscale('log')
ax[1].set_xticks(np.r_[0 : N_dirs // 2 + 1 : 5])
ax[1].set_xlabel('Configuration ID')
sns.despine()
fig.tight_layout()

In [ ]:
fmin, fmax = -6, 2
points_per_decade = 100
fname = 'Sardegna_2021_06_03cr_AC_TF_{:.1f}_{:.1f}_{}.npz'.format(fmin, fmax, points_per_decade)
TF = []
for i in tqdm(range(N_dirs)):
    data = np.load(dirs[i] / fname, allow_pickle=True)
    var_names = data['var_names'].tolist()
    idx = [n for n, name in enumerate(var_names) if re.match('.*NARCDI.*ElmTerm.fe', name)]
    assert len(idx) == 1
    TF.append(data['TF'][:, 0, idx[0]])
F = data['F']
TF = np.array(TF).T

In [ ]:
dB = 0
M = np.abs(TF)
if dB > 0:
    M = dB * np.log10(M)
phi = np.rad2deg(np.unwrap(np.angle(TF), axis=0))

In [ ]:
idx, = np.where((F >= 1e-3) & (F <= 1e2))
lw = 0.75
alpha = 0.5
fig,ax = plt.subplots(2, 2, figsize=(8,4), sharex=True)
ii, jj = np.meshgrid(idx, stable[stable % 2 == 0], indexing='ij')
ax[0, 0].plot(F[idx], M[ii, jj], 'k', lw=lw, alpha=alpha)
ax[1, 0].plot(F[idx], phi[ii, jj], 'k', lw=lw, alpha=alpha)
ii, jj = np.meshgrid(idx, unstable[unstable % 2 == 0], indexing='ij')
ax[0, 0].plot(F[idx], M[ii, jj], 'tab:red', lw=lw, alpha=alpha)
ax[1, 0].plot(F[idx], phi[ii, jj], 'tab:red', lw=lw, alpha=alpha)

ii, jj = np.meshgrid(idx, stable[stable % 2 == 1], indexing='ij')
ax[0, 1].plot(F[idx], M[ii, jj], 'k', lw=lw, alpha=alpha)
ax[1, 1].plot(F[idx], phi[ii, jj], 'k', lw=lw, alpha=alpha)
ii, jj = np.meshgrid(idx, unstable[unstable % 2 == 1], indexing='ij')
ax[0, 1].plot(F[idx], M[ii, jj], 'tab:red', lw=lw, alpha=alpha)
ax[1, 1].plot(F[idx], phi[ii, jj], 'tab:red', lw=lw, alpha=alpha)

ax[0, 0].set_xscale('log')
ax[0, 0].set_ylabel(r'$|\hat{X}(j\omega)|$')
ax[1, 0].set_ylabel(r'$\phi(X(j\omega))$')

for a in ax[-1,:]:
    a.set_xlabel('Frequency [Hz]')
    a.grid(which='major', axis='both', ls=':', lw=0.5, color=[.6,.6,.6])
for i, a in enumerate(ax[0,:]):
    a.set_title(r'$T_a = {}\,$s'.format(Ta[i]), fontsize=9)
    a.grid(which='major', axis='both', ls=':', lw=0.5, color=[.6,.6,.6])
sns.despine()
fig.tight_layout()
plt.savefig('Sardinia_TF_SM_configs.pdf')